## Indlæser masterdatasættet

In [59]:
import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf

BASE = "/Users/PC/Documents/Speciale/Analyse/Speciale"
CLEAN_DIR = os.path.join(BASE, "clean")

master = pd.read_csv(os.path.join(CLEAN_DIR, "master_monthly.csv"))
master["month"] = pd.to_datetime(master["month"])

def stjerner(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

In [ ]:
# --- NYT: USD/EUR i stedet for USD/DKK, ens valutapar på tværs af specialet ---
# EURUSD er kvoteret som USD pr. 1 EUR, så vi tager reciprokken for at få
# dollarens eget afkast (EUR pr. USD) - positivt når dollaren styrkes.
master["usd_eur_ret"] = (1 / master["EURUSD"]).pct_change()

# --- risikofrie renter, konverteret fra årlig procent (Bloomberg) til månedlig sats ---
master["r_us_m"]  = master["USD OIS 3m"] / 100 / 12
master["r_eur_m"] = master["EUR OIS 3m"] / 100 / 12

# --- excess returns iht. Campbell et al. (2010), eq. (1)/(4) ---
master["vwretd_x"]   = master["vwretd"]   - master["r_us_m"]
master["sprtrn_x"]   = master["sprtrn"]   - master["r_us_m"]
master["bond_ret_x"] = master["bond_ret"] - master["r_us_m"]
master["usd_eur_x"]  = master["usd_eur_ret"] + (master["r_us_m"] - master["r_eur_m"])

# --- kvartalsvise excess returns ---
master["vwretd_qx"]   = np.log(1 + master["vwretd_x"]).rolling(3).sum()
master["sprtrn_qx"]   = np.log(1 + master["sprtrn_x"]).rolling(3).sum()
master["bond_ret_qx"] = np.log(1 + master["bond_ret_x"]).rolling(3).sum()
master["usd_eur_qx"]  = np.log(1 + master["usd_eur_x"]).rolling(3).sum()

print("corr(usd_eur_ret, eur_ret):", master["usd_eur_ret"].corr(master["EURUSD"].pct_change()).round(3))

corr(usd_dkk_ret, eur_ret): -0.999


### Dollarens afdækningsegenskab over tid

In [ ]:
# Rullende korrelationer
win = 36
# Samme definitioner som i hedge ratio: USD/EUR og US-aktier
master["corr_usd_eq"] = master["usd_eur_ret"].rolling(win).corr(master["vwretd"])

# tjek
print(master[["month","corr_usd_eq"]].dropna().head(3))
print(master[["month","corr_usd_eq"]].dropna().tail(6))

        month  corr_usd_eq
36 2018-01-31    -0.045747
37 2018-02-28    -0.113533
38 2018-03-31    -0.073983
         month  corr_usd_eq
126 2025-07-31    -0.327869
127 2025-08-31    -0.312045
128 2025-09-30    -0.248328
129 2025-10-31    -0.246874
130 2025-11-30    -0.204327
131 2025-12-31    -0.284710


Plottet viser om dollarens negative samvariation med US-aktier ændrer sig over tid.

In [87]:
def hedge_ratio(asset_ret, fx_ret, data):
    d = data.dropna(subset=[asset_ret, fx_ret])
    cov = d[asset_ret].cov(d[fx_ret])
    var = d[fx_ret].var()
    return 1 + cov / var

print("=== h* på kvartalsvise EXCESS returns ===")
for navn, col in [("US aktier (CRSP)", "vwretd_qx"), ("S&P 500", "sprtrn_qx"), ("US statsobl.", "bond_ret_qx")]:
    h = hedge_ratio(col, "usd_dkk_qx", master)
    print(f"{navn}: h* = {h:.3f}")

def hedge_ratio_hac(asset_ret, fx_ret, data, maxlags=6):
    d = data.dropna(subset=[asset_ret, fx_ret]).copy()
    m = smf.ols(f"{asset_ret} ~ {fx_ret}", data=d).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    h_star = 1 + m.params[fx_ret]
    return h_star, m.bse[fx_ret], m.pvalues[fx_ret]

print("\n=== h* med HAC-standardfejl (kvartalsvis, EXCESS returns) ===")
for navn, col in [("US aktier (CRSP)", "vwretd_qx"), ("S&P 500", "sprtrn_qx"), ("US statsobl.", "bond_ret_qx")]:
    h, se, p = hedge_ratio_hac(col, "usd_dkk_qx", master)
    print(f"{navn}: h* = {h:.3f}{stjerner(p)}  (SE={se:.3f}, p={p:.3f})")


print("=== Sammenligning: rå vs. excess returns ===")
for navn, col_raw, col_x in [("US aktier (CRSP)", "vwretd_q", "vwretd_qx"),
                              ("S&P 500", "sprtrn_q", "sprtrn_qx"),
                              ("US statsobl.", "bond_ret_q", "bond_ret_qx")]:
    h_raw = hedge_ratio(col_raw, "usd_dkk_ret_q", master)
    h_x   = hedge_ratio(col_x, "usd_dkk_qx", master)
    print(f"{navn}: h*(rå) = {h_raw:.3f}   h*(excess) = {h_x:.3f}")

=== h* på kvartalsvise EXCESS returns ===
US aktier (CRSP): h* = 0.409
S&P 500: h* = 0.471
US statsobl.: h* = 0.714

=== h* med HAC-standardfejl (kvartalsvis, EXCESS returns) ===
US aktier (CRSP): h* = 0.409**  (SE=0.256, p=0.021)
S&P 500: h* = 0.471**  (SE=0.248, p=0.033)
US statsobl.: h* = 0.714***  (SE=0.084, p=0.001)
=== Sammenligning: rå vs. excess returns ===
US aktier (CRSP): h*(rå) = 0.409   h*(excess) = 0.409
S&P 500: h*(rå) = 0.472   h*(excess) = 0.471
US statsobl.: h*(rå) = 0.699   h*(excess) = 0.714


# TPU 

In [88]:
master["lnTPU"]   = np.log(master["TPU"])
master["lnTPU_z"] = (master["lnTPU"] - master["lnTPU"].mean()) / master["lnTPU"].std()

d = master.dropna(subset=["vwretd_qx", "usd_dkk_qx", "lnTPU_z"]).copy()
m = smf.ols("vwretd_qx ~ usd_dkk_qx * lnTPU_z", data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(m.summary().tables[1])

                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.0261      0.007      3.680      0.000       0.012       0.040
usd_dkk_qx            -0.6466      0.213     -3.030      0.002      -1.065      -0.228
lnTPU_z                0.0053      0.006      0.849      0.396      -0.007       0.018
usd_dkk_qx:lnTPU_z     0.3908      0.141      2.768      0.006       0.114       0.667


# GPR

In [93]:
# GPR (global usikkerhedsindeks)
master["lnGPR"]   = np.log(master["GPR"])
master["lnGPR_z"] = (master["lnGPR"] - master["lnGPR"].mean()) / master["lnGPR"].std()

d = master.dropna(subset=["vwretd_qx", "usd_dkk_qx", "lnGPR_z"]).copy()
m = smf.ols("vwretd_qx ~ usd_dkk_qx * lnGPR_z", data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(m.summary().tables[1])

                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.0233      0.007      3.321      0.001       0.010       0.037
usd_dkk_qx            -0.5747      0.202     -2.842      0.004      -0.971      -0.178
lnGPR_z               -0.0146      0.008     -1.765      0.078      -0.031       0.002
usd_dkk_qx:lnGPR_z     0.1179      0.196      0.603      0.546      -0.265       0.501


# GPR_USA

In [97]:
# GPRC_USA (amerikansk-specifikt geopolitisk risikoindeks)
master["lnGPR_USA"]   = np.log(master["GPRC_USA"])
master["lnGPR_USA_z"] = (master["lnGPR_USA"] - master["lnGPR_USA"].mean()) / master["lnGPR_USA"].std()

d = master.dropna(subset=["vwretd_qx", "usd_dkk_qx", "lnGPR_USA_z"]).copy()
m = smf.ols("vwretd_qx ~ usd_dkk_qx * lnGPR_USA_z", data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(m.summary().tables[1])

                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  0.0236      0.007      3.352      0.001       0.010       0.037
usd_dkk_qx                -0.5929      0.206     -2.881      0.004      -0.996      -0.190
lnGPR_USA_z               -0.0125      0.009     -1.361      0.174      -0.030       0.005
usd_dkk_qx:lnGPR_USA_z     0.1597      0.210      0.759      0.448      -0.253       0.572


# ROBUSTHEDSCHECK

In [98]:
# GPR - Threats (GPRT)
master["lnGPRT"]   = np.log(master["GPRT"])
master["lnGPRT_z"] = (master["lnGPRT"] - master["lnGPRT"].mean()) / master["lnGPRT"].std()

d = master.dropna(subset=["vwretd_q", "usd_dkk_ret_q", "lnGPRT_z"]).copy()
m = smf.ols("vwretd_q ~ usd_dkk_ret_q * lnGPRT_z", data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(m.summary().tables[1])

                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  0.0271      0.007      3.915      0.000       0.014       0.041
usd_dkk_ret_q             -0.5467      0.214     -2.550      0.011      -0.967      -0.127
lnGPRT_z                  -0.0142      0.007     -2.029      0.042      -0.028      -0.000
usd_dkk_ret_q:lnGPRT_z    -0.0834      0.176     -0.473      0.636      -0.429       0.262


In [99]:
# GPR - Acts (GPRA)
master["lnGPRA"]   = np.log(master["GPRA"])
master["lnGPRA_z"] = (master["lnGPRA"] - master["lnGPRA"].mean()) / master["lnGPRA"].std()

d = master.dropna(subset=["vwretd_q", "usd_dkk_ret_q", "lnGPRA_z"]).copy()
m = smf.ols("vwretd_q ~ usd_dkk_ret_q * lnGPRA_z", data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(m.summary().tables[1])

                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  0.0270      0.008      3.478      0.001       0.012       0.042
usd_dkk_ret_q             -0.6139      0.200     -3.075      0.002      -1.005      -0.223
lnGPRA_z                  -0.0069      0.008     -0.833      0.405      -0.023       0.009
usd_dkk_ret_q:lnGPRA_z     0.2462      0.167      1.473      0.141      -0.082       0.574
